In [0]:
from pyspark import *
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

data = [(1, 101, '2024-01-01'),
(2, 101, '2024-01-02'),
(3, 101, '2024-01-03'),
(4, 101, '2024-01-05'),
(5, 102, '2024-01-01'),
(6, 102, '2024-01-03'),
(7, 102, '2024-01-04'),
(8, 103, '2024-01-06'),
(9, 103, '2024-01-07'),
(10,103, '2024-01-08')]

df = spark.createDataFrame(data,["login_id","user_id","login_date"])
df.show()

In [0]:
window = Window.partitionBy("user_id").orderBy(col("login_date"))

rankdf = df.withColumn("rank", row_number().over(window))
rankdf.show()

In [0]:
daysdf = rankdf.withColumn("days",date_sub(col("login_date"),col("rank")))
daysdf.show()

In [0]:
finaldf = daysdf.groupBy("user_id","days").agg((count("*") >=3).alias("consecutive_days")).filter(col("consecutive_days") == 'true').drop("days","consecutive_days")
finaldf.show()